# Vishwakarma Institute of Technology, Pune
## Course: CI3201 - Deep Learning
### Lab Practical 7: Transfer Learning using Pre-trained CNN Models

---

## 1. Objective
Implement transfer learning using pre-trained **AlexNet, VGG16, ResNet50, and EfficientNetB0** models for image classification, and compare their performance using PyTorch.

In [ ]:
import time
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Load and Preprocess CIFAR-10 Dataset
# CIFAR-10 images are 32x32. We scale them up to 64x64 so AlexNet and others can process them cleanly.
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# For demonstration purposes in the lab, we can use a smaller subset if training is slow on CPU
# train_subset = torch.utils.data.Subset(trainset, range(1000))
# test_subset = torch.utils.data.Subset(testset, range(200))

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

classes = trainset.classes

In [ ]:
# 2. Define a function to build the transfer learning models
def build_model(model_name, num_classes=10):
    if model_name == 'AlexNet':
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        # Freeze features
        for param in model.features.parameters():
            param.requires_grad = False
        # Replace classifier head
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        
    elif model_name == 'VGG16':
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        for param in model.features.parameters():
            param.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        
    elif model_name == 'ResNet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'EfficientNetB0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for param in model.features.parameters():
            param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    return model.to(device)

In [ ]:
# 3. Define the training loop
def train_evaluate(model, epochs=3):
    criterion = nn.CrossEntropyLoss()
    # Only train the layers that have requires_grad=True (the new heads)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    
    history = {'train_acc': [], 'val_acc': []}
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_corrects = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
            
        train_acc = running_corrects.double() / total
        history['train_acc'].append(train_acc.item())
        
        model.eval()
        val_corrects = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += labels.size(0)
                
        val_acc = val_corrects.double() / val_total
        history['val_acc'].append(val_acc.item())
        print(f'Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')
        
    print(f"Training finished in {time.time() - start_time:.2f}s")
    return history

In [ ]:
# 4. Run Experiment across all 4 architectures
model_names = ['AlexNet', 'VGG16', 'ResNet50', 'EfficientNetB0']
results = {}

for name in model_names:
    print(f"\n--- Training {name} ---")
    model = build_model(name)
    results[name] = train_evaluate(model, epochs=3)

In [ ]:
# 5. Plot Comparison
plt.figure(figsize=(10, 6))
for name in model_names:
    plt.plot(results[name]['val_acc'], label=f'{name} (Val Acc)', marker='o')

plt.title('Transfer Learning Comparison on CIFAR-10')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()